<a href="https://colab.research.google.com/github/innoted-latam/tp2_dne_uba/blob/main/tp2_dne_uba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```
ME72: Maestría en Métodos Cuantitativos para la Gestión y Análisis de Datos
M72109: Analisis de datos no estructurados
Universidad de Buenos Aires - Facultad de Ciencias Economicas (UBA-FCE)
Año: 2026
Profesor: Facundo Santiago, Javier Ignacio Garcia Fronti
```


# Desafio de memorabilidad: Computer Vision

Como trabajo final para la materia, les proponemos resolver un desafio de memorabilidad de un video, el cual requerirá de utilizar todos los conceptos que revisamos en la materia **al mismo tiempo**: imagenes, audio y texto.

## ¿De que se trata el desafío?

Esta tarea se centra en el problema de predecir qué tan memorable es un video para un espectador. Definiremos a un video como memorable como la probabilidad de que se recuerde tal video luego de un lapso de tiempo determinado.

Recibirán un extenso conjunto de datos de videos que van acompañados de anotaciones memorables, así como predictores (features) extraídos previamente que reflejan distintos preprocesamientos de los videos para que la tarea le resulte más sencilla. Las etiquetas (labels) se ha recopilado a través de pruebas de reconocimiento y, por lo tanto, es el resultado de una medición objetiva del rendimiento de la memoria.

<img src='https://raw.githubusercontent.com/santiagxf/M72109/master/Desafio/Docs/memorability.PNG' width=600 />

*Creditos del desafio original:*

http://www.multimediaeval.org/mediaeval2019/memorability/

Mihai Gabriel Constantin, University Politehnica of Bucharest, Romania
Bogdan Ionescu, University Politehnica of Bucharest, Romania
Claire-Hélène Demarty, Technicolor, France
Quang-Khanh-Ngoc Duong, Technicolor, France
Xavier Alameda-Pineda, INRIA, France
Mats Sjöberg, CSC, Finland

Para mas información sobre esta tarea, puede revisar [Overview of The MediaEval 2021 Predicting Media Memorability Task](https://arxiv.org/abs/2112.05982)

## Direcciones

Deberán entrenar modelos de aprendizaje automático capaces de inferir la memorabilidad de video a partir del contenido **de imagenes**.

Utilizando estos datos, deberán evaluar la performance del modelo utilizando un set de datos de evaluación. Los modelos se evaluarán a través de métricas de evaluación estándar utilizadas en las tareas de clasificación y regresión (dependiendo del tipo de desafío que elijan).

Cuentan con 2 tipos de anotaciones para cada uno de los fragmentos de video disponibles:
 - **memorability_score:** Representa el puntaje de memorabilidad de la secuencia en particular, desde 0 a 1. Valores más grandes son mejores.
 - **memorable:** Variable categórica que representa si un video es memorable o no. Un video con `memorability_score` superior a `0.5` es marcado como memorable (`1`), sino es marcado como no memorable (`0`)

 > Note que aquí nuestras muestras son "secuencias" de determinadas películas. En total dispone de 660 secuencias con el nombre `sequence_name`. El mismo nombre se generó automáticamente concatenando el nombre de la pelicula a la que pertenece la secuencia (movie_name), seguido del segundo en el que comienza la secuencia, seguido del segundo en el que termina, seguido de un numero que indica el número de secuencia. Por ejemplo, la secuencia `127_hours_2000_2010_1` es un fragmento de la pelicula "127 hours", que va desde el segundo 2000 (00:33:20 hrs.) al segundo 2010 (00:33:30 hrs.) y es el fragmento número 1. Esta información es totalmente irrelevante para el problema de memorabilidad pero **el nombre de la secuencia (`sequence_name`) será su clave primeria para vincular los diferentes conjuntos de datos.**

### Entrega

Debera entregar:


2. El archivo de Colab con la solución propuesta.
3. El codigo debe:
    1. Entrenar o diseñar 1 modelo de aprendizaje automático basado en técnicas de **Computer Vision**.
    2. Poder ser ejecutable de arriba a abajo.
    3. Poder ser ejecutado sin errores.
    4. Estar claramente documentado con el paso a paso de porque realiza lo que realiza.
    5. Utilizar solo técnicas vistas en la materia.
    6. No contener código no utilizado.

4. Subir el mismo dentro de la tarea correspondiente.


## Preparación del ambiente

### Vision

In [ ]:
!pip install tensorflow tensorflow-datasets fsspec
!pip install transformers datasets evaluate diffusers accelerate
!pip matplotlib

  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### Sets de datos

Descargamos el set de datos:

In [ ]:
!wget https://raw.githubusercontent.com/santiagxf/M72109/refs/heads/master/Desafio/Data/ground_truth.csv --directory-prefix ./Data/ --quiet --no-clobber
!wget https://santiagxf.blob.core.windows.net/public/Memorability/frames.zip --directory-prefix ./Data/Features/ --quiet --no-clobber

Descompromimos los archivos

In [ ]:
!unzip -qo /content/Data/Features/frames.zip -d /content/Data/Features/frames

Los conjuntos de datos utilizados en este desafío son los siguientes:

*   `ground_truth.csv`: Contiene la verdad fundamental para el problema de clasificación. Este archivo incluye información sobre segmentos de video (`movie_name`, `start(sec)`, `end(sec)`) y su correspondiente puntuación de memorabilidad (`memorability_score`). **Aunque contiene varias columnas, para el problema de clasificación que abordaremos, la columna clave es `memorable`, que indica la probabilidad de que una persona recuerde el video.**
* Directorio `Data/Features/frames`: En lugar de trabajar con videos, hemos extraido cuadros (frames) de cada uno de los videos disponibles. Dispone de:

    *   **4 frames:** uno en el segundo 0, otro en el segundo 52, otro en el segundo 112, y un frame adicional que corresponde al que tiene la mayor cantidad de información (determinado por el método del histograma).
    *   **Nomenclatura de los frames:** Cada frame se nombra como `<sequence>__<frame>.jpg`, donde `<frame>` puede ser `0`, `52`, `112`, o `histograma`.
*   **Valores de verdad (labels):** Los valores de verdad para la memorabilidad (`memorability_score` y `memorable`) continúan estando definidos por cada secuencia completa.

**IMPORTANTE: Para proceder con el análisis, será necesario combinar estas etiquetas (`ground_truth.csv`) con los frames en el directorio /Data/Features/frames. NO ES NECESARIO que utilize los 4 frames disponibles al mismo tiempo. Puede utilizar 1, multiples, o comparar varias alternativas (por ejemplo, verificar cual de todos los frames aporta el mejor resultado).**

In [ ]:
import pandas as pd

labels = pd.read_csv('Data/ground_truth.csv')

### Tip: Preparación de datos para Computer Vision

La siguiente rutina muestra un ejemplo para recopilar las rutas de todas las imágenes de los *frames* y las asociaremos con sus `sequence_name` correspondientes del archivo `ground_truth.csv`.

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
import pandas as pd

# Directorio donde se encuentran los frames extraídos
frames_dir = '/content/Data/Features/frames/'

# Obtener todas las rutas de los archivos .jpg en el directorio de frames
all_image_paths = glob.glob(os.path.join(frames_dir, '*.jpg'))

# Extraer los 'sequence_name' de las rutas de las imágenes
# El formato de nombre de archivo es <sequence_name>__<frame_type>.jpg
image_sequence_names = [os.path.basename(path).split('__')[0] for path in all_image_paths]

# Crear un DataFrame con las rutas de las imágenes y sus sequence_name
image_df = pd.DataFrame({
    'sequence_name': image_sequence_names,
    'image_path': all_image_paths
})

# Unir con el DataFrame de labels para obtener los valores de verdad
# Nos enfocaremos en la columna 'memorable' y 'memorability_score'
dataset_df = pd.merge(image_df, labels[['sequence_name', 'memorable', 'memorability_score']], on='sequence_name', how='inner')

print(f"\nTotal de imágenes encontradas: {len(all_image_paths)}")
print(f"Total de entradas de datos con labels (tras la unión): {len(dataset_df)}")
display(dataset_df.head())


Total de imágenes encontradas: 2640
Total de entradas de datos con labels (tras la unión): 2640


,sequence_name,image_path,memorable
0,The_Lord_of_the_Rings_2_-_The_Two_Towers_4069_...,/content/Data/Features/frames/The_Lord_of_the_...,1
1,127_hours_328_338_3,/content/Data/Features/frames/127_hours_328_33...,1
2,The_Hobbit_-_An_Unexpected_Journey_6186_6196_4,/content/Data/Features/frames/The_Hobbit_-_An_...,0
3,The_Bourne_Identity_4478_4488_11,/content/Data/Features/frames/The_Bourne_Ident...,0
4,The_Green_Mile_1205_1215_41,/content/Data/Features/frames/The_Green_Mile_1...,0


## Solución